In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

path = "/Volumes/data_engineering/raw_data/raw_data/"

# check if files exist
files = dbutils.fs.ls(path)

if len(files) == 0:
    print("No new files found. Skipping Bronze ingestion.")
else:
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path + "*.csv")

    df = df.withColumn("ingestion_time", current_timestamp())

    print(f"record count:{df.count()}")

    df.write.format("delta").mode("append").option("overWriteSchema", "true").saveAsTable("data_engineering.bronze_layer.bronze_sales")